[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jfelipevasquez/Forecasting-electricity-production-Kaggle/blob/main/04_LightGBM.ipynb)

# **AVALIAÇÃO MODELO LIGHT GBM**


In [ ]:
# Librarias usadas nesse modelo
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
from lightgbm import LGBMRegressor
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import RFECV

## LEITURA E PRE-TRATAMENTO DO DATASET INICIAL

In [ ]:
#1. LEITURA INICIAL
df=pd.read_csv("https://raw.githubusercontent.com/jfelipevasquez/Forecasting-electricity-production-Kaggle/refs/heads/main/dataset/train.csv")
df_1 = df.copy()
df_1.drop(columns=['ID', 'day', 'sunrise', 'sunset','capacity_clipped'], inplace=True) ## São eliminadas colunas que não aportam ao problema

df_1['data'] = pd.to_datetime(df_1['start'])
df_1 = df_1.sort_values('data').reset_index(drop=True)

# Extração do mês(1-12) , dia (1-366) e hora (0-23)
df_1['mes'] = df_1['data'].dt.month
df_1['dia_ano'] = df_1['data'].dt.dayofyear
df_1['hora'] = df_1['data'].dt.hour

df_1.drop(columns=['start', 'time_hourly' ], inplace=True) ## Eliminação das colunas de tempo

# Transformação dos periodos de tempo em variaveis ciclicas
periodo_hora = 24
periodo_mes = 12
periodo_dia_ano = 365

# Transformación para 'hora'
df_1['hora_sin'] = np.sin(2 * np.pi * df_1['hora'] / periodo_hora)
df_1['hora_cos'] = np.cos(2 * np.pi * df_1['hora'] / periodo_hora)
# Transformación para 'mes'
df_1['mes_sin'] = np.sin(2 * np.pi * df_1['mes'] / periodo_mes)
df_1['mes_cos'] = np.cos(2 * np.pi * df_1['mes'] / periodo_mes)
# Transformación para 'dia_ano'
df_1['dia_ano_sin'] = np.sin(2 * np.pi * df_1['dia_ano'] / periodo_dia_ano)
df_1['dia_ano_cos'] = np.cos(2 * np.pi * df_1['dia_ano'] / periodo_dia_ano)

# Se é de noite (altitude solar <= 0), a produção real de treinamento deve ser 0
df_1['kw'] = np.where(df_1['altitude'] <= 0, 0, df_1['kw'])

# Não existem produções reais negativas
df_1['kw'] = np.where(df_1['kw'] < 0, 0, df_1['kw'])

# Não é possivel superar a capacidade máxima instalada (468 kW)
df_1['kw'] = np.where(df_1['kw'] > 468, 468, df_1['kw'])

# Eliminar la radição negativa  antes de criar as variaveis físicas
df_1['irradiation'] = np.where(df_1['irradiation'] < 0, 0, df_1['irradiation'])
df_1['S_d'] = np.where(df_1['S_d'] < 0, 0, df_1['S_d'])

#Radiação efetiva, Energia que o panel pode absorver (energia que cai sobre a superficie do panel * o angulo de incidencia da la luz sobre o panel)
df_1['GHI'] = df_1['irradiation'] * df_1['panel_cos']
df_1['DNI']=df_1["S_d"]*df_1["fold_cos"]

# Relaciona o momento do ano com a temperatura media
df_1['efeito_estacao_temp'] = df_1['temp_total_mean'] * df_1['dia_ano_cos']

# Índice de cobertura de nuvens densas; nuvens baixas são mais prejudiciais à produção do que nuvens altas.
df_1['nuvens_ponderadas'] = (df_1['cloud_low_mean'] * 1.0) + (df_1['cloud_mid_mean'] * 0.6) + (df_1['cloud_high_mean'] * 0.2)

# A produção diminui quando a temperatura do painel aumenta; a irradiação está relacionada à temperatura.
df_1['interacao_temp_irrad'] = df_1['irradiation'] * df_1['temp_total_mean']

# Definimos a condição com limiares modificados. Avalia quando se tem alta radiacao e producao baixa
condicao_falha = (df_1['GHI'] >= 0.80) & (df_1['kw'] <= 60)

# Removemos algumas linhas para que elas não confundissem o modelo
df_1 = df_1.drop(df_1[condicao_falha].index).reset_index(drop=True)


In [ ]:
# 2.  ANÁLISE DE CORRELAÇÃO DAS NOVAS VARIÁVEIS MACROCLIMÁTICAS
# Selecionamos as novas variáveis, a variável alvo e as principais variáveis ​​físicas de referência
vars_analise = [
    'kw', 'irradiation', 'GHI','DNI',
    'temp_total_mean','nuvens_ponderadas',
    'efeito_estacao_temp'
]
# Calcula a matriz de correlação de Pearson.
matriz_corr = df_1[vars_analise].corr()

# Configura e desenha o Mapa de Calor (Heatmap)
plt.figure(figsize=(10, 8))
sns.heatmap(
    matriz_corr,
    annot=True,          # Exibe os valores numéricos dentro de cada célula
    cmap='coolwarm',     # Escala de cores: azul (negativo), branco (zero), vermelho (positivo)
    fmt=".3f",           # Três decimais para máxima precisão.
    linewidths=0.5,      # Linha de separação sutil entre as células
    vmin=-1, vmax=1      # Limites estritos da correlação de Pearson
)

plt.title('Matriz de Correlação: Novas Variáveis ​​vs. Produção (kW)', fontsize=14, pad=20)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### Preparação das variaveis para o modelo

In [ ]:
# Características (X) incluindo as climáticas com dados faltantes
features = [
    'S_d', 'airmass', 'altitude', 'azimuth', 'irradiation', 'fold_cos', 'panel_cos',
    'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos', 'dia_ano_sin', 'dia_ano_cos',
    'temp_total_mean', 'cloud_total_mean', 'cloud_low_mean', 'cloud_high_mean', 'cloud_mid_mean',
    'GHI', 'nuvens_ponderadas', 'interacao_temp_irrad', 'DNI',
     'efeito_estacao_temp'
]

# Separar X e y
X = df_1[features]
y = df_1['kw']  # Variavel objetivo

In [ ]:
# 3. Divisão cronológica SEM CORTE
x_train, x_test, y_train, y_test= train_test_split(X,y, test_size=0.2, shuffle=False)

# Inicializa os índices para que comecem em 0
x_train = x_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
x_test = x_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

## APLICAÇÃO DO MODELO LIGHT GMB

In [ ]:
# 4. MODELO INICIAL BASELINE
LightGBM = LGBMRegressor(
    objective='regression_l1',  # Isso indica que otimiza para MAE
    max_depth=8,
    num_leaves=63,
    n_estimators=500,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.6,
    random_state=42,
    n_jobs=-1)

LightGBM.fit(x_train, y_train)

y_pred_LGB= LightGBM.predict(x_test)

mae_LGBM= mean_absolute_error(y_test, y_pred_LGB)
r2_LGBM = r2_score(y_test, y_pred_LGB)

print(f"MAE: {mae_LGBM:.2f}")
print(f"R2: {r2_LGBM:.3f}")

In [ ]:
# 5. DEFINIR VALIDADOR TEMPORAL
# AVALIAÇÃO CRUZADA Definimos um validador especial para séries temporais
tscv = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(LightGBM, x_train, y_train, scoring="neg_mean_absolute_error", cv=tscv)
LGB_mae_scores = (-scores).round(3)

MAE_mean_LGB = np.mean(LGB_mae_scores.T)
print('MAE de cada Fold temporal:', LGB_mae_scores)
print('Promedio real del MAE temporal:', MAE_mean_LGB)


#FEATURE SELECTION ANTES DE HIPERPARAMETROS


##Feature importance

In [ ]:
# 6. Aplicação de Feature importance
# Extrair o ganho nativo do modelo
importancias_gain = LightGBM.booster_.feature_importance(importance_type='gain')

# Crear el DataFrame y ordenar de mayor a menor
df_importancia = pd.DataFrame({
    'Variavel': features,
    'Ganho': importancias_gain
}).sort_values(by='Ganho', ascending=False).reset_index(drop=True)

df_importancia['Ganho'] = (df_importancia['Ganho'] / df_importancia['Ganho'].sum() * 100).map('{:.2f}%'.format)

# Mostrar o ranking
df_importancia

##Permutation importance

In [ ]:
# 7. Calcular a importancia pura (sklearn entrega o incremento do MAE em positivo)
resultado_perm = permutation_importance(
    LightGBM,
    x_test,
    y_test,
    scoring='neg_mean_absolute_error',
    random_state=42, n_jobs=-1
)

# Criar o DataFrame e ordenar de maior a menor impacto real
df_perm = pd.DataFrame({
    'Variavel': features,
    'Impacto_MAE': resultado_perm.importances_mean
}).sort_values(by='Impacto_MAE', ascending=False).reset_index(drop=True)

# Transformar diretamente a porcentagem sobre a mesma columna
df_perm['Impacto_MAE'] = (df_perm['Impacto_MAE'] / df_perm['Impacto_MAE'].sum() * 100).map('{:.2f}%'.format)

df_perm

##RFECV

In [ ]:
# 8. Configurar o selector
#mean_feature_selector ajuda a no ficar com poucas
selector=RFECV(
    estimator=LightGBM,
    step=1,cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
    )

#Ajustar a los datos
selector.fit(x_train, y_train)

#Ver cuales variables sobreviven
features_seleccionadas=X.columns[selector.support_]
print(f"Variables recomendadas ({len(features_seleccionadas)}): {list(features_seleccionadas)}")

## Otimizar fluxo com RFECV e Permutation

In [ ]:
# 9. Filtrar o DataSet
features_ot=features_seleccionadas.tolist()

# Filtrar os sets de dados atuais usando o x_test e o x_train anteriores
x_train_ot=x_train[features_ot]
x_test_ot=x_test[features_ot]

In [ ]:
# 10. MODELO BASELINE COM VARIAVEIS REDUZIDAS
LightGBM_OT= LGBMRegressor(
    objective='regression_l1',  # Isso indica que otimiza para MAE
    max_depth=8,
    num_leaves=63,
    n_estimators=500,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.6,
    random_state=42,
    n_jobs=-1)

# Treinar o modelo otimizado
LightGBM_OT.fit(x_train_ot, y_train)

# Avaliar o set do test
y_pred_LGB_ot= LightGBM_OT.predict(x_test_ot)
mae_LGBM_ot= mean_absolute_error(y_test, y_pred_LGB_ot)
r2_LGBM_ot = r2_score(y_test, y_pred_LGB_ot)

print(f"MAE depois do RFCEV: {mae_LGBM_ot:.2f}")
print(f"R2 depois do RFCEV: {r2_LGBM_ot:.3f}")


In [ ]:
# 11. Validação cruzada temporária para o novo conjunto
scores_rfecv = cross_val_score(LightGBM_OT, x_train_ot, y_train, scoring="neg_mean_absolute_error", cv=tscv)
mae_cv_rfecv_mean = (-scores_rfecv).mean()
print('MAE de cada Fold temporal:', scores_rfecv)
print('Promedio real do MAE temporal:', mae_cv_rfecv_mean)

In [ ]:
# =============================================================================
# TABELA COMPARATIVA DE RESULTADOS
# =============================================================================
print("\n" + "="*50)
print("       TABELA DE RENDIMIENTO: ANTES VS. DEPOIS")
print("="*50)
print(f"Métrica               | Modelo Original (23 vars) | Modelo RFECV (16 vars)")
print(f"----------------------|---------------------------|------------------------")
print(f"MAE Test (Inmediato)  | {mae_LGBM:.2f}                     | {mae_LGBM_ot:.2f}")
print(f"MAE CV (Promedio Temp)| {MAE_mean_LGB:.2f}                     | {mae_cv_rfecv_mean:.2f}")
print("="*50)

#Modelo sem variaveis que não contribuem


##Entrenando otra vez el modelo

In [ ]:
# 12. BUSCA DE HIPERPARAMETROS

import optuna
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Estabelecer a função que Optuna vai maximizar/minimizar
def objective(trial): ##Define a função de optuna que vai executar

# define faixas (más acotados para evitar overfitting)
    param_space = {
        'objective': 'regression_l1', # Métrica MAE
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'n_estimators': 1500,         # Alto, porque o early stopping vai parar antes

        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True), ## PEde o optuna que sugura um ritmo de aprendizagem nessa faixa

        'max_depth': trial.suggest_int('max_depth', 4, 8),       # Arvore mais curtos = más rápidos
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),   # Menos folhas para evitar decorar

        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),

        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),

        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    # Lista para acumular el MAE de cada fold temporal
    fold_maes = []

    # Recorrer a validação cruzada temporal (tscv)
    for train_idx, val_idx in tscv.split(x_train_ot):

        # Separar train e validação para ESTE fold
        X_tr, X_va = x_train_ot.iloc[train_idx], x_train_ot.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Inicializar LightGBM con os parámetros sugeridos nesta vez
        model = lgb.LGBMRegressor(**param_space)

        # treinar usando a PACIENCIA (Early Stopping)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        # Predezir no bloco de validação e salvar o erro
        preds = model.predict(X_va)
        fold_maes.append(mean_absolute_error(y_va, preds))

    # Optuna procurara minimizar a media do MAE dos folds
    return sum(fold_maes) / len(fold_maes)

# Começar o estudo Bayesiano
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30) # 30 iteraões inteligentes equivalen a >100 aleatorias

# Mostrar os melhores resultados
print(f"Melhor MAE media em CV Temporal: {study.best_value:.2f}")
print("\nMelhores Hiperparámetros Encontrados:")
for param, valor in study.best_params.items():
    print(f"  -> {param}: {valor}")

# treinar o modelo final com os ganhadores enm TODO o set de treinamento
best_params_final = study.best_params
best_params_final['objective'] = 'regression_l1'
best_params_final['n_estimators'] = 500 # Valor seguro e definitivo para o test final

best_model_opt = lgb.LGBMRegressor(**best_params_final, random_state=42, n_jobs=-1)
best_model_opt.fit(x_train_ot, y_train)

# Avaliação final no set de test
y_pred_opt = best_model_opt.predict(x_test_ot)
mae_LGBM_opt = mean_absolute_error(y_test, y_pred_opt)
r2_LGBM_opt = r2_score(y_test, y_pred_opt)

print(f"MAE Optimizada: {mae_LGBM_opt:.2f}")
print(f"R2 Optimizada: {r2_LGBM_opt:.3f}")

# CURVAS DE APRENDIZAGEM


In [ ]:
# 13. Curvas de apredizagem
# Inicializar listas para armazenar no processo
muestras_entrenamiento = []
train_maes = []
val_maes = []

# Iterar sobre as janelas de tempo reais para o crescimento do dataset
for train_idx, val_idx in tscv.split(x_train_ot):

    # Separar conjuntos de treinamento e validação para o fold atual
    X_tr, X_va = x_train_ot.iloc[train_idx], x_train_ot.iloc[val_idx]
    y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Inicializar o LightGBM com os melhores parámetros encontrados por Optuna
    # Usamos um n_estimators fixo // não aplicamos early stopping aquí
    params_curva = best_params_final.copy()
    params_curva['n_estimators'] = 500

    model_fold = lgb.LGBMRegressor(**params_curva, random_state=42, n_jobs=-1)

    # treinaer el modelo com a janela de tempo cumulada até este fold
    model_fold.fit(X_tr, y_tr)

    # Realizar predições em ambos sets
    preds_train = model_fold.predict(X_tr)
    preds_val = model_fold.predict(X_va)

    # Salvar o tamanho do set de treinamento e os  MAE
    muestras_entrenamiento.append(len(X_tr))
    train_maes.append(mean_absolute_error(y_tr, preds_train))
    val_maes.append(mean_absolute_error(y_va, preds_val))

# Plotar a curva de aprendizagem temporalt
plt.figure(figsize=(10, 5))

#Curva de erro de treino
plt.plot(muestras_entrenamiento, train_maes, marker='o', color='#b22222',
         linewidth=2, label='Erro de Treino (Train MAE)')

# Curva de erro de validação (Verde)
plt.plot(muestras_entrenamiento, val_maes, marker='o', color='#228b22',
         linewidth=2, label='Erro de Validação (Val MAE)')

# Configurações  da curva
plt.title('Curva de Aprendizagem Real - Modelo Otimizado',
          fontsize=12, fontweight='bold')
plt.xlabel('Mostras de Treinamento Acumulado no Tempo', fontsize=10)
plt.ylabel('MAE (Menor é melhor)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=10)
plt.tight_layout()

# Mostrar gráfico
plt.show()